In [ ]:
import numpy as np
import pandas as pd
import yaml
import re
import json

from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
df_firms = pd.read_csv(dataset_config['path_processed'] + 'WOS/POST00_US_firms_filtered.csv')
df_firms['type'] = 'f'
df_firms

In [ ]:
df_types = pd.read_csv(dataset_config['path_processed'] + 'WOS/02_WOS_unique_orgs_with_type.csv')
df_types

In [ ]:
df_university = df_types[(df_types.country=='US') & (df_types.type=='Education')].drop(columns=['country'])
df_university['type'] = 'u'
df_university

In [ ]:
df_firms_uni = pd.concat([df_firms, df_university]).rename(columns={'organization': 'organizations'})
df_firms_uni

In [ ]:
df_wos = pd.read_csv(dataset_config['path_processed'] + 'WOS/WOS_2010_2023.csv')
df_wos

In [ ]:
df_filtered = df_wos.merge(df_firms_uni)
df_filtered

In [ ]:
df_filtered.type.value_counts()

In [ ]:
df_canonical = pd.read_parquet(dataset_config['path_processed'] + 'WOS/04_canonical_firms_US.parquet').rename(columns={'organization': 'organizations'})[['organizations', 'canonical_affiliation']]
df_canonical

In [ ]:
df_filtered = df_filtered.merge(df_canonical, how='left')
df_filtered

In [ ]:
df_filtered['affiliationame'] = df_filtered['organizations']

In [ ]:
df_filtered[df_filtered.type == 'f'].canonical_affiliation.isna().value_counts() 
# True: means these papers are published by MNCs in China

In [ ]:
wos_classification = df_filtered[
    ~((df_filtered['type'] == 'f') & (df_filtered['canonical_affiliation'].isna()))
]
# Override canonical_affiliation with affiliationame if it already has a value
wos_classification.loc[wos_classification['canonical_affiliation'].notna(), 'affiliationame'] = wos_classification['canonical_affiliation']
wos_classification

In [ ]:
wos_classification.drop(columns=['organizations', 'canonical_affiliation'], inplace=True)
wos_classification

## Merge Affiliation Clusters

In [ ]:
import json

file_path = dataset_config['path_processed'] + 'WOS/05_affiliation_clusters_US.json'

with open(file_path, 'r', encoding='utf-8') as f:
    cluster_data = json.load(f)

mapping = {}

for cluster in cluster_data:
    if len(cluster) > 1:  # skip singletons
        shortest = min(cluster, key=len)  # find shortest string
        for item in cluster:
            mapping[item] = shortest

print(len(mapping))

In [ ]:
wos_classification['affiliationame'] = (wos_classification['affiliationame'].apply(lambda x: mapping.get(x, x)))
wos_classification

In [ ]:
wos_classification[['wosid', 'affiliationame', 'type']].to_parquet(dataset_config['path_processed'] + 'WOS/POST01_WOS_paper_US.parquet')